# Food Spoilage Detection — Exploratory Analysis

This notebook walks through the full pipeline interactively.

In [ ]:
import os, sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

## 1. Load & inspect dataset

In [ ]:
from data_loader import get_generators, describe_dataset

train_gen, val_gen, test_gen = get_generators()
describe_dataset(train_gen, 'Train')
describe_dataset(val_gen,   'Val')
describe_dataset(test_gen,  'Test')

## 2. Visualise sample images

In [ ]:
from config import CLASSES

images, labels = next(train_gen)
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for i, ax in enumerate(axes.flatten()):
    if i >= len(images): break
    ax.imshow(images[i])
    ax.set_title(CLASSES[np.argmax(labels[i])], fontsize=9)
    ax.axis('off')
plt.suptitle('Augmented training samples', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Build model and summarise

In [ ]:
from model import build_model
model, base_model = build_model()
model.summary(line_length=90)

## 4. Train (Phase 1 — feature extraction)

In [ ]:
from config import EPOCHS
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from data_loader import get_class_weights

cw = get_class_weights(train_gen)

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    class_weight=cw,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, verbose=1),
    ],
)

## 5. Plot training curves

In [ ]:
h = history.history
ep = range(1, len(h['accuracy']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(ep, h['accuracy'], label='Train'); ax1.plot(ep, h['val_accuracy'], label='Val')
ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(True)
ax2.plot(ep, h['loss'],     label='Train'); ax2.plot(ep, h['val_loss'],     label='Val')
ax2.set_title('Loss');     ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()

## 6. Evaluate on test set

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

y_prob = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = test_gen.classes

print(classification_report(y_true, y_pred, target_names=CLASSES))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
plt.tight_layout(); plt.show()

## 7. Run real-time prediction on a single image

In [ ]:
from data_loader import preprocess_single_image

# Replace with any image path on your machine
IMAGE_PATH = '../data/test/fresh/your_image.jpg'

if os.path.exists(IMAGE_PATH):
    inp   = preprocess_single_image(IMAGE_PATH)
    probs = model.predict(inp, verbose=0)[0]
    pred  = CLASSES[np.argmax(probs)]
    conf  = np.max(probs)

    plt.imshow(plt.imread(IMAGE_PATH))
    plt.title(f'Prediction: {pred}  ({conf*100:.1f}%)', fontsize=13)
    plt.axis('off')
    plt.show()
else:
    print('Set IMAGE_PATH to a valid image file.')